
# Improving Customer Satisfaction through Automated Review Sentiment Analysis  
### NLP Assignment – July 2025 Batch

## Overview
As a product manager at an e-commerce company, the objective of this project is to build an **automated sentiment analysis system** that classifies customer reviews as **Positive** or **Negative**.  
This system helps in monitoring product performance, identifying negative feedback early, and improving customer satisfaction.



## Dataset Description
- **Rows:** 10,000 customer reviews  
- **Columns:**
  1. **label** – Sentiment of the review (`pos` or `neg`)
  2. **review** – Actual text review written by the customer


In [ ]:

# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from wordcloud import WordCloud

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

nltk.download('stopwords')


## Load Dataset

In [ ]:

df = pd.read_csv('amazonreviews.tsv', sep='\t')
df.head()


## Data Overview

In [ ]:

df.info()
df['label'].value_counts()


## Data Cleaning and Text Preprocessing

In [ ]:

df.drop_duplicates(inplace=True)
df.dropna(inplace=True)

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def clean_text(text):
    text = text.lower()
    text = ''.join([char for char in text if char.isalpha() or char == ' '])
    words = text.split()
    words = [stemmer.stem(word) for word in words if word not in stop_words]
    return ' '.join(words)

df['clean_review'] = df['review'].apply(clean_text)
df.head()


## Label Encoding

In [ ]:

df['sentiment'] = df['label'].map({'pos': 1, 'neg': 0})
df.head()


## Exploratory Data Analysis

In [ ]:

sns.countplot(x='sentiment', data=df)
plt.xticks([0,1], ['Negative', 'Positive'])
plt.title("Sentiment Distribution")
plt.show()


In [ ]:

positive_text = ' '.join(df[df['sentiment'] == 1]['clean_review'])
negative_text = ' '.join(df[df['sentiment'] == 0]['clean_review'])

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.imshow(WordCloud(background_color='white').generate(positive_text))
plt.axis('off')
plt.title("Positive Reviews Word Cloud")

plt.subplot(1,2,2)
plt.imshow(WordCloud(background_color='black').generate(negative_text))
plt.axis('off')
plt.title("Negative Reviews Word Cloud")

plt.show()


## Train-Test Split and Feature Extraction

In [ ]:

X = df['clean_review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)


## Model Development – Logistic Regression

In [ ]:

model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)


## Model Evaluation

In [ ]:

y_pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))


In [ ]:

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()


## Cross-Validation

In [ ]:

cv_scores = cross_val_score(model, X_train_tfidf, y_train, cv=5, scoring='f1')
print("Cross-validation F1 scores:", cv_scores)
print("Average F1 score:", cv_scores.mean())



## Business Insights and Conclusion
- The sentiment analysis model successfully classifies customer reviews into positive and negative categories.
- Negative reviews can be monitored in real time to detect problematic products.
- Helps customer support teams take faster action on complaints.
- TF-IDF with Logistic Regression provides a strong and interpretable baseline model.
